# Station Characterization and Profiling

This notebook explores station usage patterns to identify distinct profiles (weekly peaks, weekend-heavy, etc.). We use K-Means clustering to group stations with similar temporal signatures.

In [8]:
import os
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import MinMaxScaler

# Add project root to path
PROJECT_ROOT = Path(os.getcwd()).parents[0]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from backend.database.db_io import connect_db
from dotenv import load_dotenv

# Load environment variables from project root .env
load_dotenv(PROJECT_ROOT / '.env')

conn = connect_db()
city_id = 1 # Madrid example
print(f"Connected to database for city_id: {city_id}")

Connected to database for city_id: 1


## 1. Data Loading

We fetch station readings (availability) and station metadata (locations).

In [9]:
query = """
    SELECT 
        r.station_id, 
        s.name,
        s.lat,
        s.lon,
        r.observed_at, 
        r.available_bikes,
        r.available_bikes - LAG(r.available_bikes) OVER (PARTITION BY r.station_id ORDER BY r.observed_at) as delta
    FROM station_readings r
    JOIN stations s ON s.city_id = r.city_id AND s.station_id = r.station_id
    WHERE r.city_id = %s
    AND r.observed_at > NOW() - INTERVAL '30 days'
"""
df = pd.read_sql(query, conn, params=(city_id,))
df['observed_at'] = pd.to_datetime(df['observed_at'])
df['hour'] = df['observed_at'].dt.hour
df['day_of_week'] = df['observed_at'].dt.dayofweek
df['is_weekend'] = df['day_of_week'] >= 5

print(f"Loaded {len(df)} readings for {df['station_id'].nunique()} stations.")

Loaded 0 readings for 0 stations.


/var/folders/q_/kvl1vqd97qq_hn08p839948m0000gn/T/ipykernel_11658/2076528498.py:15: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn, params=(city_id,))


## 2. Feature Engineering

We want to cluster stations based on *when* they are used, regardless of their total capacity. 
We define 'trips' as the absolute change in availability (proxy for arrivals + departures).

In [10]:
# Define trips (positive delta = return, negative delta = unlock)
df['trips'] = df['delta'].abs()

# Group by station, hour, and weekend status
profile = df.groupby(['station_id', 'hour', 'is_weekend'])['trips'].mean().unstack(level=[1, 2])

# Clean up column names (Hour_Weekend-Status)
profile.columns = [f"H{h}_{'WE' if we else 'WD'}" for h, we in profile.columns]
profile = profile.fillna(0)

# Normalize each station's profile (Min-Max) so we compare shapes
scaler = MinMaxScaler()
profile_norm = pd.DataFrame(
    scaler.fit_transform(profile.T).T, 
    index=profile.index, 
    columns=profile.columns
)

profile_norm.head()

ValueError: at least one array or dtype is required

## 3. Clustering Analysis

Determining the optimal number of clusters (Elbow Method).

In [ ]:
inertia = []
K_range = range(1, 11)
for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(profile_norm)
    inertia.append(kmeans.inertia_)

plt.figure(figsize=(8, 4))
plt.plot(K_range, inertia, 'bx-')
plt.xlabel('Number of Clusters (k)')
plt.ylabel('Inertia')
plt.title('The Elbow Method showing the optimal k')
plt.show()

### Run K-Means with Optimal K


In [ ]:
k_optimal = 4 # Based on expected profiles for bike-sharing (Commuter vs Leisure)
kmeans = KMeans(n_clusters=k_optimal, random_state=42, n_init=10)
profile['cluster'] = kmeans.fit_predict(profile_norm)
profile_norm['cluster'] = profile['cluster']

print(f"Clustered {len(profile)} stations into {k_optimal} profiles.")

## 4. Profile Interpretation

Visualizing the centroids to understand what each cluster represents.

In [ ]:
centroids = profile_norm.groupby('cluster').mean()

fig, axes = plt.subplots(2, 2, figsize=(15, 10), sharey=True)
axes = axes.flatten()

for i in range(k_optimal):
    c_data = centroids.iloc[i]
    wd = [c_data[f"H{h}_WD"] for h in range(24)]
    we = [c_data[f"H{h}_WE"] for h in range(24)]
    
    axes[i].plot(range(24), wd, label='Weekday', linewidth=2)
    axes[i].plot(range(24), we, label='Weekend', linestyle='--')
    axes[i].set_title(f"Cluster {i}")
    axes[i].set_xticks(range(0, 24, 4))
    axes[i].legend()

plt.tight_layout()
plt.show()

## 5. Spatial Distribution

Mapping the stations colored by cluster.

In [ ]:
# Merge cluster info back with coordinates
geo_data = df[['station_id', 'name', 'lat', 'lon']].drop_duplicates().set_index('station_id')
geo_data['cluster'] = profile['cluster']

plt.figure(figsize=(10, 10))
sns.scatterplot(data=geo_data, x='lon', y='lat', hue='cluster', palette='viridis', s=100, alpha=0.7)
plt.title('Station Clusters Spatial Distribution')
plt.axis('equal')
plt.show()